# similar  —  Sec 3.1  cosine perturbation / genuine-vs-noise

Frozen Qwen2.5-1.5B, split-context SQuAD. Anchors are re-measured
in-run (floor = reader's own masked cache, reference = its unmasked cache).

Results are written to `results/`, figures to `figures/`.


# Does a VERY SIMILAR (high-cosine) injected cache work? — training-free geometry test

The death experiment: E-SR-D cache cos≈0.007 → unreadable. Heavy-alignment: trained cos≈0.95 → still
unreadable. Here we skip E-SR-D entirely and hand the **frozen reader** a cache we construct to be
genuinely high-cosine to a real one, then just decode.

- **Arm A** — reader's OWN cache + calibrated noise to hit a target cosine; sweep the target. *How
  faithful must a cache be before the reader breaks?* (pure geometry; content has no fact → tracks floor.)
- **Arm B** — interpolate own→holder (holder has the fact); sweep the blend, log achieved cosine AND
  accuracy. *Does a small, high-cosine nudge toward the fact transfer anything?*

In [ ]:
import os
# Write next to this notebook's parent (the repo root), not the CWD, so
# results land in the same results/ that make_figs.py and verify_results.py read.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
os.makedirs("results", exist_ok=True)
os.makedirs("figures", exist_ok=True)
import os
CACHE = os.environ.get("HF_HOME") or os.path.expanduser("~/.cache/huggingface")
os.environ["HF_HOME"]=CACHE; os.environ["HF_DATASETS_CACHE"]=CACHE
os.environ["TOKENIZERS_PARALLELISM"]="false"
import json, numpy as np, re
import torch, torch.nn as nn, torch.nn.functional as F
import datasets
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.cache_utils import DynamicCache
import matplotlib.pyplot as plt
try:
    get_ipython().run_line_magic("matplotlib","inline")
except NameError:
    pass  # running as a plain script, not under IPython
assert torch.cuda.is_available(), (
    "This experiment needs a CUDA GPU. The reader runs in fp32 (deliberately: the\n"
    "cosine and attention-logit measurements are sensitive to reduced precision),\n"
    "so expect ~8 GB and more for the larger notebooks.")
DEVICE="cuda"; SEED=0; torch.manual_seed(SEED); np.random.seed(SEED)
print("Device:",torch.cuda.get_device_name(0),flush=True)
BASE="Qwen/Qwen2.5-1.5B"
CFG=dict(n_eval=200, max_ctx=256, gen_len=8,
         cos_targets=[0.9999,0.999,0.995,0.99,0.95,0.90],
         alphas=[0.0,0.1,0.25,0.5,0.75,1.0])
if os.environ.get("KVL_FAST","0")=="1":
    CFG.update(n_eval=24); print(">>> FAST <<<",flush=True)
# A fast run writes to a separate tag so it cannot overwrite the
# committed paper results.
_default_tag = "_similarfast" if os.environ.get("KVL_FAST","0")=="1" else "_similar"
TAG=os.environ.get("KVL_TAG", _default_tag)
print("CFG",CFG,"TAG",TAG,flush=True)

## 1. Load frozen base; probe GQA

In [ ]:
tok=AutoTokenizer.from_pretrained(BASE,cache_dir=CACHE)
if tok.pad_token is None: tok.pad_token=tok.eos_token
base=AutoModelForCausalLM.from_pretrained(BASE,cache_dir=CACHE,torch_dtype=torch.float32).to(DEVICE).eval()
for p in base.parameters(): p.requires_grad_(False)
ids=tok("hi",return_tensors="pt").input_ids.to(DEVICE)
with torch.no_grad(): pkv=base(ids,use_cache=True).past_key_values
N_LAYER=len(pkv); _,N_KV,_,D_HEAD=pkv[0][0].shape; KV_DIM=2*N_KV*D_HEAD
print(f"L={N_LAYER} n_kv={N_KV} d_head={D_HEAD} kv_dim={KV_DIM}",flush=True)

## 2. Cache tooling (identical to the death experiment) + reserved-last-token decode

In [ ]:
def kv_stack(past):
    outs=[]
    for (k,v) in past:
        B,H,T,dh=k.shape
        outs.append(torch.cat([k.permute(0,2,1,3).reshape(B,T,H*dh),
                               v.permute(0,2,1,3).reshape(B,T,H*dh)],-1))
    return torch.stack(outs,0)                       # (L,B,T,kv_dim)
def native_ids(text,max_len):
    return tok([text],return_tensors="pt",truncation=True,max_length=max_len).input_ids.to(DEVICE)
@torch.no_grad()
def ctx_cache(text,max_len):
    ids=native_ids(text,max_len); last=ids[:,-1:]; body=ids[:,:-1]
    leg=base(body,use_cache=True).past_key_values
    if not isinstance(leg,tuple): leg=leg.to_legacy_cache() if hasattr(leg,"to_legacy_cache") else leg
    return kv_stack(leg), last
def slots_to_legacy(vec):
    L,B,Kk,_=vec.shape; half=N_KV*D_HEAD; leg=[]
    for l in range(L):
        kt=vec[l,:,:,:half].reshape(B,Kk,N_KV,D_HEAD).permute(0,2,1,3).contiguous()
        vt=vec[l,:,:,half:].reshape(B,Kk,N_KV,D_HEAD).permute(0,2,1,3).contiguous()
        leg.append((kt,vt))
    return leg
def to_past(vec): return DynamicCache.from_legacy_cache(tuple(slots_to_legacy(vec)))
def _pos(s,l): return torch.arange(s,s+l,device=DEVICE).unsqueeze(0)
@torch.no_grad()
def greedy_from_prefix(prefix_vec,last_id):
    past=to_past(prefix_vec); T0=prefix_vec.shape[2]
    o=base(last_id,past_key_values=past,position_ids=_pos(T0,1),
           attention_mask=torch.ones(1,T0+1,device=DEVICE),use_cache=True)
    past=o.past_key_values; cur=o.logits[:,-1,:].argmax(-1,keepdim=True); gen=[cur]; L=T0+1
    for _ in range(CFG["gen_len"]-1):
        o=base(cur,past_key_values=past,position_ids=_pos(L,1),
               attention_mask=torch.ones(1,L+1,device=DEVICE),use_cache=True)
        past=o.past_key_values; cur=o.logits[:,-1,:].argmax(-1,keepdim=True); gen.append(cur); L+=1
    return tok.decode(torch.cat(gen,1)[0])

## 3. Split-context SQuAD (holder full / reader masked) — same as death experiment

In [ ]:
def make_prompt(p,q): return f"{p}\nQuestion: {q}\nAnswer:"
def mask_answer(p,a):
    i=p.find(a); return None if i<0 else p[:i]+"____"+p[i+len(a):]
def build_qa(n,split):
    ds=datasets.load_dataset("squad",split=split,cache_dir=CACHE); items=[]
    for ex in ds:
        a=ex["answers"]["text"][0].strip()
        if not a or a not in ex["context"] or len(a.split())>4: continue
        m=mask_answer(ex["context"],a)
        if m is None: continue
        items.append(dict(q=ex["question"].strip(),full=ex["context"].strip(),masked=m.strip(),ans=a))
        if len(items)>=n: break
    return items
eval_qa=build_qa(CFG["n_eval"],"validation")
print("eval",len(eval_qa),flush=True)
def norm(s): return re.sub(r"[^a-z0-9 ]","",s.lower()).strip()
def hit(t,g): return norm(g) in norm(t) if g else False
@torch.no_grad()
def reader_self_kv(it): return ctx_cache(make_prompt(it["masked"],it["q"]),CFG["max_ctx"])
@torch.no_grad()
def reader_full_kv(it): return ctx_cache(make_prompt(it["full"],it["q"]),CFG["max_ctx"])
@torch.no_grad()
def holder_full_kv(it): return ctx_cache(make_prompt(it["full"],it["q"]),CFG["max_ctx"])
def cos_full(a,b): return F.cosine_similarity(a.reshape(-1),b.reshape(-1),dim=0).item()

## 4. Arm A — noisy copy of OWN cache, calibrated to a target cosine

For a target cosine `c`, a copy `x' = x + sigma*n` with unit-var noise `n` has (in expectation over the
whole flattened tensor) `cos(x,x') ≈ 1/sqrt(1 + sigma^2 * E[n^2]/E[x^2])`. We pick sigma per-item to hit
the target, verify the ACHIEVED cosine empirically, decode, and record accuracy. Content = the reader's
own masked cache (NO fact), so this measures ONLY the frozen reader's tolerance to geometric perturbation.

In [ ]:
@torch.no_grad()
def noisy_to_cos(vec, target):
    # solve sigma so that cos(vec, vec+sigma*noise) ~= target, per whole tensor
    n=torch.randn_like(vec)
    # scale noise to unit power relative to signal
    sig_pow=(vec**2).mean().sqrt(); n=n/ (n**2).mean().sqrt() * sig_pow
    # cos ~= 1/sqrt(1+r^2) with r = ||sigma*n||/||vec|| = sigma  (since powers matched) -> sigma=sqrt(1/c^2-1)
    r=float((1.0/target**2 - 1.0)**0.5)
    return vec + r*n

@torch.no_grad()
def arm_A(data):
    out={f"cos{c}":0 for c in CFG["cos_targets"]}; out["floor"]=0; out["ceiling"]=0
    achieved={f"cos{c}":[] for c in CFG["cos_targets"]}; n=0
    for it in data:
        g=it["ans"]
        own,last=reader_self_kv(it)
        full,lastf=reader_full_kv(it)
        out["floor"]  += hit(greedy_from_prefix(own,last),g)
        out["ceiling"]+= hit(greedy_from_prefix(full,lastf),g)
        for c in CFG["cos_targets"]:
            cc=noisy_to_cos(own,c); achieved[f"cos{c}"].append(cos_full(own,cc))
            out[f"cos{c}"] += hit(greedy_from_prefix(cc,last),g)
        n+=1
    acc={k:v/max(1,n) for k,v in out.items()}
    ach={k:float(np.mean(v)) for k,v in achieved.items()}
    return acc,ach
accA,achA=arm_A(eval_qa)
print("\n=== ARM A: noisy copy of OWN cache (content has NO fact) ===",flush=True)
print(f"  floor (own, no noise)   = {accA['floor']:.3f}",flush=True)
for c in CFG["cos_targets"]:
    print(f"  target cos {c:<7} (achieved {achA[f'cos{c}']:.4f})  acc = {accA[f'cos{c}']:.3f}",flush=True)
print(f"  ceiling (own full)      = {accA['ceiling']:.3f}",flush=True)

## 5. Arm B — interpolate own→holder (holder has the FACT)

`c = (1-a)*own + a*holder` over the overlapping positions (min length; remaining positions stay own).
Sweep `a`. We log achieved `cos(c, own)` and accuracy. a=0 is floor; a=1 injects the holder's fact-bearing
cache directly. If any intermediate a (high cosine to own) beats floor, a similar-but-fact-tilted cache
transferred the fact.

In [ ]:
@torch.no_grad()
def blend(own, holder, a):
    # align on min length over the position axis (dim=2); blend those, keep own elsewhere
    S=min(own.shape[2], holder.shape[2])
    c=own.clone()
    c[:,:,:S,:]=(1-a)*own[:,:,:S,:]+a*holder[:,:,:S,:]
    return c
@torch.no_grad()
def arm_B(data):
    out={f"a{a}":0 for a in CFG["alphas"]}; ach={f"a{a}":[] for a in CFG["alphas"]}; n=0
    for it in data:
        g=it["ans"]
        own,last=reader_self_kv(it)
        holder,_=holder_full_kv(it)
        for a in CFG["alphas"]:
            c=blend(own,holder,a); ach[f"a{a}"].append(cos_full(own,c))
            out[f"a{a}"] += hit(greedy_from_prefix(c,last),g)
        n+=1
    acc={k:v/max(1,n) for k,v in out.items()}
    achm={k:float(np.mean(v)) for k,v in ach.items()}
    return acc,achm
accB,achB=arm_B(eval_qa)
print("\n=== ARM B: interpolate own->holder (holder HAS the fact) ===",flush=True)
for a in CFG["alphas"]:
    print(f"  a={a:<4} cos(c,own)={achB[f'a{a}']:.4f}  acc={accB[f'a{a}']:.3f}"
          + ("   <- floor" if a==0.0 else "   <- raw holder inject" if a==1.0 else ""),flush=True)

## 6. Plots + save + verdict

In [ ]:
RES=dict(config=CFG,base=BASE,armA=dict(acc=accA,achieved=achA),armB=dict(acc=accB,achieved=achB))
json.dump(RES,open(f"results/similar{TAG}.json","w"),indent=1)

fig,ax=plt.subplots(1,2,figsize=(12,4.3))
# Arm A: acc vs achieved cosine
cxs=[achA[f'cos{c}'] for c in CFG["cos_targets"]]; cys=[accA[f'cos{c}'] for c in CFG["cos_targets"]]
ax[0].plot(cxs,cys,'-o',color='#2c6fbb')
ax[0].axhline(accA['floor'],ls='--',color='#7f8c8d',label=f"floor={accA['floor']:.3f}")
ax[0].axhline(accA['ceiling'],ls='--',color='#2e8b57',label=f"ceiling={accA['ceiling']:.3f}")
ax[0].set_xlabel("achieved cosine to own cache"); ax[0].set_ylabel("accuracy")
ax[0].set_title("Arm A: reader tolerance to geometric noise\n(content = own, no fact)"); ax[0].invert_xaxis()
ax[0].legend(); ax[0].grid(ls=':')
# Arm B: acc vs alpha, twin cosine
axb=ax[1]; xs=CFG["alphas"]; ys=[accB[f'a{a}'] for a in xs]
axb.plot(xs,ys,'-o',color='#c0392b',label='accuracy')
axb.axhline(accA['floor'],ls='--',color='#7f8c8d',label=f"floor={accA['floor']:.3f}")
axb.set_xlabel("blend a  (0=own, 1=holder)"); axb.set_ylabel("accuracy"); axb.grid(ls=':')
ax2=axb.twinx(); ax2.plot(xs,[achB[f'a{a}'] for a in xs],'-s',color='#e08e0b',alpha=.6,label='cos(c,own)')
ax2.set_ylabel("cos(c,own)",color='#e08e0b')
axb.set_title("Arm B: nudge own->holder (holder has fact)"); axb.legend(loc='upper right')
fig.tight_layout(); fig.savefig("figures/similar_curves.png",dpi=150); plt.show()

print("\n=== VERDICT ===",flush=True)
# Arm A: find the highest cosine at which acc already dropped well below floor
tol=[c for c in CFG["cos_targets"] if accA[f'cos{c}']>=accA['floor']-0.02]
print(f"Arm A: reader keeps ~floor accuracy down to cosine >= "
      f"{min([achA[f'cos{c}'] for c in tol]) if tol else 'NONE (breaks even at 0.9999)':}",flush=True)
bestB=max(CFG["alphas"], key=lambda a: accB[f'a{a}'])
print(f"Arm B: best blend a={bestB} acc={accB[f'a{bestB}']:.3f} vs floor={accA['floor']:.3f} "
      f"-> {'BEATS floor (transfer!)' if accB[f'a{bestB}']>accA['floor']+0.02 else 'no transfer'}",flush=True)
print("saved.",flush=True)